In [1]:
import os

import numpy as np
import pandas as pd

In [2]:
raw_path = 'data/raw'
processed_path = 'data/processed'

In [3]:
temperature_specific_heat_df = pd.DataFrame()

# JGR Earth Surface - 2018 - Liu - Thermal Regime and Properties of Soils and Ice‐Rich Permafrost in Beacon Valley

In [4]:
df_4 = pd.read_csv(f'{raw_path}/F2_Temperature_vs_Specific_Heat_Capacity.csv')

In [5]:
df_4

,Temperature_(C),Specific_Heat_(Jkg^-K^-1)
0,-47.91854,590.13378
1,-39.99377,605.43478
2,-20.01290,641.55518
3,-30.00337,623.74582
4,-10.02236,658.86288
5,-0.03172,675.41806
6,7.91452,687.95987


In [6]:
df_4.columns = df_4.columns.str.strip()
df_4['Temperature_(C)'] = pd.to_numeric(df_4['Temperature_(C)'], errors='coerce')
df_4['Specific_Heat_(Jkg^-1K^-1)'] = pd.to_numeric(df_4['Specific_Heat_(Jkg^-K^-1)'], errors='coerce')
df_4 = df_4[['Temperature_(C)', 'Specific_Heat_(Jkg^-1K^-1)']].sort_values('Temperature_(C)').reset_index(drop=True)

In [7]:
new_rows = df_4.rename(columns={
    'Temperature_(C)': 'Temperature',
    'Specific_Heat_(Jkg^-1K^-1)': 'Specific_Heat'
})

new_rows['Soil'] = np.nan
new_rows['Source'] = 'Liu et al. 2018'
new_rows = new_rows[['Temperature', 'Specific_Heat', 'Soil', 'Source']]

In [8]:
temperature_specific_heat_df = pd.concat([temperature_specific_heat_df, new_rows], ignore_index=True)

In [9]:
temperature_specific_heat_df.tail()

,Temperature,Specific_Heat,Soil,Source
2,-30.00337,623.74582,NaN,Liu et al. 2018
3,-20.01290,641.55518,NaN,Liu et al. 2018
4,-10.02236,658.86288,NaN,Liu et al. 2018
5,-0.03172,675.41806,NaN,Liu et al. 2018
6,7.91452,687.95987,NaN,Liu et al. 2018


# Temperature-dependentSpecficHeat_Kay and Goit_1975

In [10]:
df_5 = pd.read_csv(f'{raw_path}/F1_Temperature_vs_Specific_Heat.csv')

In [11]:
df_5

,Temperature_(K),Specific_Heat_(caldeg^-1g^-1),Soil
0,201.88463,0.14430,Clay
1,222.24458,0.14602,Clay
2,241.97148,0.16289,Clay
3,251.63392,0.17150,Clay
4,261.55764,0.18103,Clay
5,272.31438,0.19386,Clay
6,281.14516,0.20463,Clay
7,300.16219,0.22117,Clay
8,232.61767,0.15963,Clay
9,211.54824,0.14219,Clay


In [12]:
df_5.columns = df_5.columns.str.strip()
df_5['Temperature_(K)'] = pd.to_numeric(df_5['Temperature_(K)'], errors='coerce')
df_5['Specific_Heat_(caldeg^-1g^-1)'] = pd.to_numeric(df_5['Specific_Heat_(caldeg^-1g^-1)'], errors='coerce')
df_5['Soil'] = df_5['Soil'].str.strip()

# Convert K to C and cal g^-1 K^-1 to J kg^-1 K^-1.
df_5['Temperature_(C)'] = df_5['Temperature_(K)'] - 273.15
df_5['Specific_Heat_(Jkg^-1K^-1)'] = df_5['Specific_Heat_(caldeg^-1g^-1)'] * 4184
df_5 = df_5[['Temperature_(C)', 'Specific_Heat_(Jkg^-1K^-1)', 'Soil']].sort_values(['Soil', 'Temperature_(C)']).reset_index(drop=True)

In [13]:
new_rows = df_5.rename(columns={
    'Temperature_(C)': 'Temperature',
    'Specific_Heat_(Jkg^-1K^-1)': 'Specific_Heat'
})

new_rows['Source'] = 'Kay and Goit 1975'
new_rows = new_rows[['Temperature', 'Specific_Heat', 'Soil', 'Source']]

In [14]:
temperature_specific_heat_df = pd.concat([temperature_specific_heat_df, new_rows], ignore_index=True)

In [15]:
temperature_specific_heat_df.tail()

,Temperature,Specific_Heat,Soil,Source
43,-11.27739,667.59904,Sand,Kay and Goit 1975
44,-2.06596,698.56064,Sand,Kay and Goit 1975
45,7.89285,719.22960,Sand,Kay and Goit 1975
46,16.82595,727.89048,Sand,Kay and Goit 1975
47,26.92032,745.25408,Sand,Kay and Goit 1975


## Standardize For Modeling

In [16]:
temperature_specific_heat_df = temperature_specific_heat_df.copy()
for column in ['Temperature', 'Specific_Heat', 'Soil', 'Source']:
    if column not in temperature_specific_heat_df.columns:
        temperature_specific_heat_df[column] = np.nan

temperature_specific_heat_df['Soil'] = temperature_specific_heat_df['Soil'].fillna('Unknown')

temperature_cp_model_df = temperature_specific_heat_df.rename(columns={
    'Specific_Heat': 'target_value',
    'Temperature': 'temperature_c',
    'Soil': 'soil_class',
    'Source': 'source'
}).copy()
temperature_cp_model_df['water_content'] = np.nan
temperature_cp_model_df['water_content_basis'] = np.nan
temperature_cp_model_df['bulk_density'] = np.nan
temperature_cp_model_df['data_type'] = 'Observed'
temperature_cp_model_df['dataset_family'] = 'temperature_based'
temperature_cp_model_df['target_type'] = 'specific_heat'
temperature_cp_model_df = temperature_cp_model_df[[
    'target_value', 'target_type', 'dataset_family',
    'water_content', 'water_content_basis', 'temperature_c', 'bulk_density',
    'soil_class', 'data_type', 'source'
]]

temperature_cp_model_df.head()

,target_value,target_type,dataset_family,water_content,water_content_basis,temperature_c,bulk_density,soil_class,data_type,source
0,590.13378,specific_heat,temperature_based,NaN,NaN,-47.91854,NaN,Unknown,Observed,Liu et al. 2018
1,605.43478,specific_heat,temperature_based,NaN,NaN,-39.99377,NaN,Unknown,Observed,Liu et al. 2018
2,623.74582,specific_heat,temperature_based,NaN,NaN,-30.00337,NaN,Unknown,Observed,Liu et al. 2018
3,641.55518,specific_heat,temperature_based,NaN,NaN,-20.01290,NaN,Unknown,Observed,Liu et al. 2018
4,658.86288,specific_heat,temperature_based,NaN,NaN,-10.02236,NaN,Unknown,Observed,Liu et al. 2018


In [17]:
os.makedirs(processed_path, exist_ok=True)
temperature_specific_heat_df.to_csv(
    f'{processed_path}/temperature_specific_heat_cleaned.csv',
    index=False,
)
temperature_cp_model_df.to_csv(
    f'{processed_path}/temperature_cp_model_df.csv',
    index=False,
)